# 04 - Advanced CNN with Data Augmentation

Pushing toward 99.5%+ by:
1. **Data augmentation** — random affine transforms (shifts, rotations, scaling) create training variety
2. **Deeper architecture** — 3 conv layers for richer feature extraction
3. **ReduceLROnPlateau** — adaptive learning rate

In [1]:
import numpy as np
import duckdb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path
from tqdm.notebook import tqdm

## 1. Load Data

In [2]:
con = duckdb.connect(str(Path('../data/mnist.duckdb')), read_only=True)
train_data = con.execute("SELECT * FROM mnist WHERE split='train'").fetchnumpy()
test_data = con.execute("SELECT * FROM mnist WHERE split='test'").fetchnumpy()
con.close()

px_cols = [f'px{i}' for i in range(784)]
X_train = np.column_stack([train_data[c] for c in px_cols]).astype(np.float32) / 255.0
y_train = train_data['label'].astype(np.int64)
X_test = np.column_stack([test_data[c] for c in px_cols]).astype(np.float32) / 255.0
y_test = test_data['label'].astype(np.int64)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (60000, 784), Test: (10000, 784)


## 2. Augmented Dataset

Data augmentation applies random transforms to training images each epoch, effectively giving the model new examples every pass. This reduces overfitting and improves generalization.

In [3]:
class AugmentedMNIST(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = torch.tensor(images).reshape(-1, 1, 28, 28)
        self.labels = torch.tensor(labels)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_transform = T.Compose([
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
])

BATCH_SIZE = 128
train_ds = AugmentedMNIST(X_train, y_train, transform=train_transform)
test_ds = AugmentedMNIST(X_test, y_test, transform=None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

## 3. Deeper CNN (3 Conv Layers)

```
Input (1, 28, 28)
→ Conv(1→32, 3×3) + BN + ReLU
→ Conv(32→64, 3×3) + BN + ReLU + MaxPool(2×2)  → (64, 13, 13)
→ Conv(64→128, 3×3) + BN + ReLU + MaxPool(2×2) → (128, 5, 5) [with padding]
→ Flatten → Linear(3200→256) + ReLU + Dropout(0.4)
→ Linear(256→10)
```

In [4]:
class MNISTDeepCNN(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 4. Training

In [5]:
EPOCHS = 25
LEARNING_RATE = 1e-3

model = MNISTDeepCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

Model parameters: 1,275,594
MNISTDeepCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in tqdm(range(EPOCHS), desc='Training'):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(y_batch)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    history['train_loss'].append(running_loss / total)
    history['train_acc'].append(correct / total)

    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * len(y_batch)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += len(y_batch)
    history['test_loss'].append(running_loss / total)
    history['test_acc'].append(correct / total)

    scheduler.step(history['test_loss'][-1])

    print(f"Epoch {epoch+1:2d} | Train Loss: {history['train_loss'][-1]:.4f} Acc: {history['train_acc'][-1]:.4f} | "
          f"Test Loss: {history['test_loss'][-1]:.4f} Acc: {history['test_acc'][-1]:.4f}")

Training:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch  1 | Train Loss: 0.3115 Acc: 0.9043 | Test Loss: 0.0535 Acc: 0.9830
Epoch  2 | Train Loss: 0.1197 Acc: 0.9638 | Test Loss: 0.0305 Acc: 0.9899
Epoch  3 | Train Loss: 0.0959 Acc: 0.9709 | Test Loss: 0.0364 Acc: 0.9881
Epoch  4 | Train Loss: 0.0823 Acc: 0.9744 | Test Loss: 0.0212 Acc: 0.9922
Epoch  5 | Train Loss: 0.0755 Acc: 0.9773 | Test Loss: 0.0165 Acc: 0.9943
Epoch  6 | Train Loss: 0.0682 Acc: 0.9790 | Test Loss: 0.0242 Acc: 0.9919
Epoch  7 | Train Loss: 0.0647 Acc: 0.9807 | Test Loss: 0.0208 Acc: 0.9932
Epoch  8 | Train Loss: 0.0623 Acc: 0.9815 | Test Loss: 0.0166 Acc: 0.9946
Epoch  9 | Train Loss: 0.0557 Acc: 0.9833 | Test Loss: 0.0188 Acc: 0.9938
Epoch 10 | Train Loss: 0.0410 Acc: 0.9880 | Test Loss: 0.0136 Acc: 0.9955
Epoch 11 | Train Loss: 0.0403 Acc: 0.9882 | Test Loss: 0.0131 Acc: 0.9956


## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'], label='Test')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curve'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(y_batch)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Deep CNN + Augmentation — Confusion Matrix')
plt.show()

print(classification_report(all_labels, all_preds, digits=4))

## 7. Save Model

In [ ]:
import pandas as pd

# Save training history to DuckDB
history_df = pd.DataFrame(history)
history_df['epoch'] = range(1, EPOCHS + 1)

con = duckdb.connect(str(Path('../data/mnist.duckdb')))
con.execute('DROP TABLE IF EXISTS deep_cnn_training_history')
con.execute('CREATE TABLE deep_cnn_training_history AS SELECT * FROM history_df')
con.close()

# Save model weights
torch.save(model.state_dict(), '../data/mnist_deep_cnn_model.pth')
print(f'Deep CNN model saved. Final test accuracy: {history["test_acc"][-1]:.4f}')